In [1]:
import numpy as np
from typing import List, Union

alpha_j = 0.5          
a, b, n = 0.5, 1.5, 10
h = (b - a) / n        

def f(x):
    return alpha_j * np.exp(x) + (1 - alpha_j) * np.cos(x)

nodes  = np.array([a + i * h for i in range(n + 1)])
values = f(nodes)

x_star   = nodes[0]    + (2 / 3) * h   # x_0 + 2h/3
x_star2  = nodes[n//2] + (1 / 2) * h   # x_{n/2} + h/2
x_star3  = nodes[n]    - (1 / 3) * h   # x_n - h/3
restore_points = np.array([x_star, x_star2, x_star3])
restore_values = f(restore_points)

## 3. Интерполяционный многочлен Ньютона

Разделённая разность нулевого порядка совпадает со значением $f(x_i)$. Разделённая разность $(k+1)$-го порядка определяется формулой: $$f(x_0, \dots, x_{k+1}) = \frac{f(x_1,\dots, x_{k+1}) - f(x_0, \dots, x_k)}{x_{k+1}- x_0}$$

Интерполяционный многочлен Ньютона: $$P_n(x) = f(x_0) + (x-x_0) \cdot f(x_0, x_1) + (x-x_0)(x-x_1) \cdot f(x_0, x_1, x_2) + \dots (x-x_0) \dots (x-x_{n-1}) \cdot f(x_0, \dots, x_n)$$

In [2]:
def divided_differences(x, y):
    n = len(x)
    div_diff = y.copy()
    
    for j in range(1, n):
        for i in range(n - 1, j - 1, -1):
            div_diff[i] = (div_diff[i] - div_diff[i - 1]) / (x[i] - x[i - j])
    
    return div_diff

def newton_polynomial(x_points, y_points, x):
    coeffs = divided_differences(x_points, y_points)
    n = len(coeffs)
    
    x = np.asarray(x, dtype=float)
    result = np.full_like(x, coeffs[0], dtype=float)
    product = np.ones_like(x, dtype=float)
    
    for k in range(1, n):
        product *= (x - x_points[k - 1])
        result += coeffs[k] * product
        
    return result.item() if result.ndim == 0 else result

newton = newton_polynomial(nodes, values, restore_points)
np.abs(newton - restore_values)

array([8.34887715e-14, 2.22044605e-15, 1.90070182e-13])

## 4. Остаток интерполирования в форме Лагранжа



$$|r_n(x)| \le |\omega_{n+1}(x)| \frac{\max_{x \in [a,b]} |f^{(n+1)}(x)|}{(n+1)!}$$

$$f^{(n+1)}(x) = f^{(11)}(x) = 0.5 e^x + 0.5 \sin x$$

$$\max f^{(11)}(x) = 0.5 e^{1.5} + 0.5 \sin 1.5$$

In [7]:
import math

def omega_n1(x, x_points):
    x = np.asarray(x, dtype=float)
    prod = np.ones_like(x, dtype=float)
    for xi in x_points:
        prod *= (x - xi)
    return prod

def newton_remainder_bound(x,
                           x_points,
                           max_deriv,
                           n):
    fact = math.factorial(n + 1)
    omega = omega_n1(x, x_points)
    return np.abs(omega) * max_deriv / fact

max_f11 = 0.5 * math.exp(b) + 0.5 * math.sin(b)
newton_remainder_bound(restore_points, nodes, max_f11, n)

array([1.32045029e-13, 3.29197498e-15, 2.81439584e-13])